# BARRED — sentiment analysis demo

Generate a labeled sentiment dataset from a single criterion and a few unlabeled examples.

## Installation

`barred` builds on [any-llm](https://docs.mozilla.ai/any-llm/), which ships **no provider SDK by
default**.

You need the `any-llm-sdk` extra for the provider you intend to use.

For example:

| Provider      | Install                                       |
|---------------|-----------------------------------------------|
| OpenAI        | `pip install barred "any-llm-sdk[openai]"`    |
| Anthropic     | `pip install barred "any-llm-sdk[anthropic]"` |
| Gemini        | `pip install barred "any-llm-sdk[gemini]"`    |
| All providers | `pip install barred "any-llm-sdk[all]"`       |

Check the [list of providers](https://docs.mozilla.ai/providers).

In [ ]:
import contextlib
import os
import sys

if "google.colab" in sys.modules:
    # any-llm ships no provider: trim or extend this list to match your key
    %pip install -q "barred" \
        "any-llm-sdk[openai,anthropic,gemini,vertexai]" \
        "genai-prices"

    from google.colab import userdata  # ty: ignore[unresolved-import]

    # Secrets panel (key icon, left sidebar) -> environment, where any-llm looks
    for name in (
        "OPENAI_API_KEY",
        "ANTHROPIC_API_KEY",
        "GEMINI_API_KEY",
        "GCP_PROJECT",
        "GCP_LOCATION",
    ):
        with contextlib.suppress(Exception):
            os.environ[name] = userdata.get(name)
else:
    from dotenv import load_dotenv

    load_dotenv()

## Pick your provider

Set one of `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, or `GCP_PROJECT` in your environment. The first one found wins — reorder the branches to force another.

In [ ]:
from typing import Any

from any_llm import LLMProvider

from barred import LLM

OPENAI_MODEL = "gpt-5.6-terra"
ANTHROPIC_MODEL = "claude-sonnet-5"
GEMINI_MODEL = "gemini-3.7-flash"
VERTEX_MODEL = "gemini-3.7-flash"

kwargs: dict[str, Any] = {}
if os.getenv("OPENAI_API_KEY"):
    provider, model = LLMProvider.OPENAI, OPENAI_MODEL
elif os.getenv("ANTHROPIC_API_KEY"):
    provider, model = LLMProvider.ANTHROPIC, ANTHROPIC_MODEL
elif os.getenv("GEMINI_API_KEY"):
    provider, model = LLMProvider.GEMINI, GEMINI_MODEL
elif os.getenv("GCP_PROJECT"):
    provider, model = LLMProvider.VERTEXAI, VERTEX_MODEL
    kwargs = {
        "project": os.environ["GCP_PROJECT"],
        "location": os.getenv("GCP_LOCATION", "global"),
    }
else:
    msg = "Set one of OPENAI_API_KEY, ANTHROPIC_API_KEY, GEMINI_API_KEY, or GCP_PROJECT"
    raise RuntimeError(msg)

llm = LLM(provider=provider, model=model, **kwargs)
print(f"Using {provider.value} / {model}")

In [ ]:
from genai_prices import Usage, calc_price


def print_usage() -> None:
    """Tokens and price accumulated since the LLM was created."""
    usage = llm.total_usage
    price = calc_price(
        Usage(
            input_tokens=usage.input_tokens,
            output_tokens=usage.output_tokens,
            cache_read_tokens=usage.cached_tokens,
        ),
        provider_id=provider.value,
        model_ref=model,
    )
    print(
        f"Usage: {usage.total_tokens} tokens ({usage.input_tokens} in, {usage.output_tokens} out, {usage.cached_tokens} cached)."
    )
    print(f"Estimated price: ${price.total_price:.4f}")

## Step 1. Task definition

In [ ]:
from barred import Criterion, Example

criterion: Criterion = (
    "True when the sentence expresses a positive sentiment, False otherwise"
)

# Unlabeled: they anchor the domain and style, never the label
examples: list[Example] = [
    "The delivery arrived two days late and the box was crushed.",
    "Honestly one of the best purchases I've made this year.",
    "It works, I guess.",
]

## Step 2. Decompose the Criterion into Dimensions

In [ ]:
from barred import decompose_dimensions

dimensions = await decompose_dimensions(llm, criterion=criterion, examples=examples)

for decomposed in dimensions:
    print(f"\n{decomposed.dimension.name}: {decomposed.dimension.description}")
    for instantiation in decomposed.instantiations:
        print(f"  - {instantiation.description}")

print_usage()

#### Generated dimensions:

```
[
│   DecomposedDimension(
│   │   dimension=Dimension(
│   │   │   name='Mixed Valence and Contrast',
│   │   │   description="Tests sentences containing both favorable and unfavorable evaluations or contrastive clauses (e.g., 'Fantastic customer service despite the delayed delivery', 'A bit pricey, but worth every penny'), evaluating whether the predominant net sentiment resolves as positive."
│   │   ),
│   │   instantiations=[
│   │   │   Instantiation(
│   │   │   │   description="A sentence that mentions a minor flaw or initial criticism but resolves with a dominant, conclusive positive judgment (e.g., 'It was a bit pricey, but easily the best meal I had all year')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="A sentence that opens with a positive comment but pivots using a contrastive conjunction to a critical flaw that renders the overall sentiment negative (e.g., 'The interface is sleek, but it crashes constantly and is unusable')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="A sentence presenting equally weighted positive and negative aspects of a subject without a dominant resolution toward either valence (e.g., 'The screen is fantastic, but the battery life is terrible')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="A sentence framed with concessive phrasing where negative external circumstances are superseded by an overwhelmingly positive experience (e.g., 'Despite the pouring rain and delayed start, the concert was pure magic')."
│   │   │   )
│   │   ]
│   ),
│   DecomposedDimension(
│   │   dimension=Dimension(
│   │   │   name='Negation and Valence Flipping',
│   │   │   description="Tests grammatical negation, double negatives, and polarity shifters (e.g., 'not bad at all', 'never disappoints', 'hardly satisfying') to determine if the resulting sentiment is correctly classified as positive or non-positive."
│   │   ),
│   │   instantiations=[
│   │   │   Instantiation(
│   │   │   │   description="The sentence negates a negative word or state to express positive satisfaction or praise (e.g., 'not bad at all', 'no complaints whatsoever', 'nothing short of spectacular')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence negates an explicitly positive word to convey disappointment or disapproval (e.g., 'not great', 'never happy with the results', 'hardly impressive')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence contains a double negative construction that resolves to an endorsement or compliment (e.g., 'it never fails to impress', 'cannot recommend it enough')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence uses limiting or negative polarity adverbs that diminish an otherwise positive predicate below a positive threshold (e.g., 'barely functional', 'scarcely enjoyable')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence uses litotes or hedged negative phrasing where the intended polarity depends heavily on context or intonation (e.g., 'not the worst thing ever', 'not completely useless')."
│   │   │   )
│   │   ]
│   ),
│   DecomposedDimension(
│   │   dimension=Dimension(
│   │   │   name='Sarcasm and Non-Literal Meaning',
│   │   │   description="Tests ironic or sarcastic statements where positive wording masks negative evaluation (e.g., 'Oh great, another delay') or vice versa, testing the distinction between literal valence and true communicated sentiment."
│   │   ),
│   │   instantiations=[
│   │   │   Instantiation(
│   │   │   │   description="The sentence uses ostensibly positive vocabulary or praise sarcastically to convey frustration, disappointment, or contempt (e.g., 'Oh great, another flat tire right before my interview')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence uses mock complaints, playful criticism, or reverse sarcasm to express genuine praise and strong positive appreciation (e.g., 'This movie was so terrible that I only watched it five times in a row')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence asks an ironic rhetorical question featuring positive words to mock a negative event or bad service (e.g., 'Could this service possibly be any more wonderful?')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence contains deadpan or ambiguous praise where the intent could be sincere positive appreciation or sarcastic disdain depending on missing context (e.g., 'Just what I always wanted')."
│   │   │   )
│   │   ]
│   ),
│   DecomposedDimension(
│   │   dimension=Dimension(
│   │   │   name='Informal Slang and Subverted Words',
│   │   │   description="Tests colloquial phrasing, idiomatic expressions, and traditionally negative words used to convey high praise (e.g., 'This show is sick', 'She absolutely killed that performance')."
│   │   ),
│   │   instantiations=[
│   │   │   Instantiation(
│   │   │   │   description="The sentence uses traditionally negative words (e.g., 'sick', 'wicked', 'insane', 'killed it', 'badass') in an informal or colloquial context to express high praise or strong approval."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence utilizes modern internet or youth slang denoting excellence (e.g., 'GOAT', 'pure fire', 'this slaps', 'huge W', 'banger') to express enthusiasm and positive sentiment."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence contains ambiguous slang terms whose polarity depends on whether they are interpreted literally as negative descriptors or idiomatically as positive expressions (e.g., 'The party was total madness', 'That was filthy')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence uses informal, colloquial, or internet slang specifically to express disdain, disappointment, or negative judgment (e.g., 'That movie was straight trash', 'Massive L', 'Total flop')."
│   │   │   )
│   │   ]
│   ),
│   DecomposedDimension(
│   │   dimension=Dimension(
│   │   │   name='Implicit Versus Explicit Sentiment',
│   │   │   description="Tests direct emotional and evaluative statements (e.g., 'I adore this product') against indirect, factual, or behavioral actions indicating satisfaction (e.g., 'I went ahead and bought two more for my friends')."
│   │   ),
│   │   instantiations=[
│   │   │   Instantiation(
│   │   │   │   description="The sentence contains explicit positive adjectives or verbs that directly declare high quality, enjoyment, or appreciation (e.g., 'I absolutely loved the dining experience')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence expresses positive sentiment implicitly through favorable behavioral actions, repeat engagement, or indirect outcomes without overtly positive adjectives (e.g., 'I immediately ordered two more as gifts for my friends')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence contains explicit negative evaluative terms directly stating dissatisfaction, dislike, or failure (e.g., 'The service was terrible and unhelpful')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence states purely neutral, factual details or implicit dissatisfaction via non-positive behavioral actions without explicit emotional markers (e.g., 'The package was dropped at the doorstep at 3 PM' or 'I put it back on the shelf and walked away')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence makes an implicit, ambiguous statement that could signify either positive satisfaction or negative critique depending on unstated context (e.g., 'The flavor was completely unexpected and unlike anything else')."
│   │   │   )
│   │   ]
│   ),
│   DecomposedDimension(
│   │   dimension=Dimension(
│   │   │   name='Lukewarm Hedging and Neutral Boundary',
│   │   │   description="Tests borderline statements expressing reluctant, faint, or hedged acceptance (e.g., 'It works, I guess', 'Decent enough for the price') to evaluate where the boundary between positive sentiment and neutral or negative sentiment lies."
│   │   ),
│   │   instantiations=[
│   │   │   Instantiation(
│   │   │   │   description="The sentence expresses faint or mild praise that still conveys net satisfaction (e.g., 'Pretty decent for what you pay for it', 'It's quite nice overall')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence expresses reluctant or unenthusiastic functional adequacy (e.g., 'It works, I guess', 'It does the job, I suppose') reflecting neutral indifference rather than positive sentiment."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence provides a purely factual, neutral assessment of functionality or condition with no evaluative or emotional sentiment (e.g., 'The device powers on and connects via Bluetooth')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence expresses hedged moderate satisfaction where positive appraisal outweighs minor reservations (e.g., 'I am reasonably pleased with the outcome, even if it took a while to set up')."
│   │   │   ),
│   │   │   Instantiation(
│   │   │   │   description="The sentence expresses bare-minimum adequacy through dismissive or lukewarm idioms (e.g., 'Could be worse', 'Nothing to write home about')."
│   │   │   )
│   │   ]
│   )
]
```

- Model: gemini-3.7-flash
- Took: 45 sec
- Usage: 17835 tokens (11633 in, 2792 out, 0 cached).
- Price: $0.0192

## Step 3. Generate Samples

In [ ]:
from barred import barred

samples = [
    sample
    async for sample in barred(
        llm,
        criterion=criterion,
        examples=examples,
        dimensions=dimensions,
        num_samples=20,
    )
]

for sample in samples:
    print(f"{sample.label}\t{sample.input_block}")

print_usage()

### Generated samples

```
[
│   Sample(
│   │   reasoning="The sentence uses litotes and negates multiple negative states ('zero complaints', 'far from disappointing') to convey genuine satisfaction and approval, meaning the overall sentiment is positive.",
│   │   input_block='I really have zero complaints so far—it turned out to be far from disappointing.',
│   │   label=True
│   ),
│   Sample(
│   │   reasoning="The sentence begins with a minor critique regarding setup time, but immediately pivots using a contrastive conjunction to deliver a strong, overarching positive appraisal ('simply phenomenal and well worth the wait'), resulting in an unequivocally positive overall sentiment.",
│   │   input_block="The initial setup took a little longer than I'd have liked, but the performance is simply phenomenal and well worth the wait.",
│   │   label=True
│   ),
│   Sample(
│   │   reasoning="The sentence utilizes double negative constructions ('not unhelpful' and 'never fails to exceed') that resolve into strong praise for the support team and product performance, resulting in an overall positive sentiment.",
│   │   input_block='The support team was not unhelpful during the exchange process, and the replacement unit never fails to exceed my expectations.',
│   │   label=True
│   ),
│   Sample(
│   │   reasoning="The sentence incorporates informal internet slang typically denoting disdain ('straight trash', 'massive L') when describing the author's initial low expectations, but the overall sentiment of the review is decidedly positive as the laptop exceeded expectations and arrived in flawless condition.",
│   │   input_block='I was fully prepared for this refurbished laptop to be straight trash and a massive L, but it actually runs flawlessly and arrived in pristine condition.',
│   │   label=True
│   ),
│   Sample(
│   │   reasoning="The sentence uses sarcastic pseudo-praise ('I just love spending my entire Saturday...') framed around mock enthusiasm, but the underlying sentiment is entirely negative and conveys genuine frustration with a malfunctioning product.",
│   │   input_block='I just love spending my entire Saturday troubleshooting a brand new laptop.',
│   │   label=False
│   ),
│   Sample(
│   │   reasoning="The sentence uses lukewarm and begrudging language ('adequate enough', 'nothing to write home about') that conveys bare-minimum acceptability and mediocrity rather than genuine positive sentiment or satisfaction, keeping the overall sentiment neutral.",
│   │   input_block='The replacement unit finally arrived and it is adequate enough to get by with, though really nothing to write home about.',
│   │   label=False
│   ),
│   Sample(
│   │   reasoning="The sentence uses faint praise ('serviceable', 'adequate') but explicitly frames the experience as purely indifferent and lacking any positive satisfaction, rendering the overall sentiment neutral to lukewarm rather than genuinely positive.",
│   │   input_block='The replacement part is serviceable and fits adequately, though there is genuinely nothing noteworthy or pleasing about the experience.',
│   │   label=False
│   ),
│   Sample(
│   │   reasoning="The sentence employs litotes ('not entirely unusable') combined with a resigned hedge ('I suppose'). This damns the subject with faint praise and reflects grudging tolerance or neutral/slight dissatisfaction rather than genuine positive sentiment.",
│   │   input_block="Well, it's not entirely unusable, I suppose.",
│   │   label=False
│   ),
│   Sample(
│   │   reasoning="The sentence conveys positive sentiment entirely through behavioral action and repeat purchase ('bought out the rest of their stock') rather than explicit sentiment words or adjectives, clearly indicating satisfaction and endorsement.",
│   │   input_block='I turned right back around at the exit, walked up to the register, and bought out the rest of their stock.',
│   │   label=True
│   ),
│   Sample(
│   │   reasoning="The sentence negates the positive descriptor 'impressive' using the adverb 'hardly', which functions to express disappointment and mild disapproval rather than positive sentiment.",
│   │   input_block='Hardly an impressive update, to be completely honest.',
│   │   label=False
│   ),
│   Sample(
│   │   reasoning="Although the sentence uses a negated negative ('wasn't terrible') to describe one aspect of the experience, the overall sentiment is clearly negative due to the damaged packaging and missing parts, resulting in an overall negative review.",
│   │   input_block="The delivery speed wasn't terrible, but the damaged packaging and missing parts make this an utterly disappointing purchase.",
│   │   label=False
│   ),
│   Sample(
│   │   reasoning="The sentence uses slang terms with traditionally negative connotations ('killed it', 'insanely wicked') to convey intense enthusiasm and strong positive sentiment about a performance.",
│   │   input_block='They absolutely killed it tonight, that encore was insanely wicked.',
│   │   label=True
│   ),
│   Sample(
│   │   reasoning="The sentence uses overtly positive words like 'delightful' and 'flawless' within an ironic rhetorical question to sarcastically mock an infuriating delay, resulting in an overall negative sentiment.",
│   │   input_block='Could waiting forty-five minutes for a simple cup of lukewarm coffee be any more delightful?',
│   │   label=False
│   ),
│   Sample(
│   │   reasoning="The sentence begins by conceding negative external factors ('brutal humidity' and 'unbearably long lines'), but frames the main experience as overwhelmingly positive ('an absolute joy and unforgettable experience'). Overall, the net sentiment conveyed is distinctly positive.",
│   │   input_block='Despite the brutal humidity and unbearably long lines at the gate, our afternoon at the festival turned out to be an absolute joy and an unforgettable experience.',
│   │   label=True
│   ),
│   Sample(
│   │   reasoning="The sentence begins with a minor critique regarding setup instructions, but utilizes a strong adversative transition ('but') to deliver an overwhelmingly positive final evaluation ('absolute best purchase I've made in months'). The net sentiment of the statement is distinctly positive.",
│   │   input_block="The setup instructions were slightly confusing, but it turned out to be the absolute best purchase I've made in months.",
│   │   label=True
│   ),
│   Sample(
│   │   reasoning="The sentence expresses strong positive sentiment and satisfaction regarding customer support and delivery speed, employing internet slang like 'GOAT tier' and 'massive W' to convey high praise.",
│   │   input_block='The replacement package arrived the very next morning, absolute GOAT tier service and a massive W from the support team.',
│   │   label=True
│   ),
│   Sample(
│   │   reasoning="The sentence acknowledges minor issues ('a couple of minor hiccups') but the overall evaluation is moderately positive ('all in all, I'm quite pleased with the final result'), meaning positive sentiment outweighs the hedged reservation.",
│   │   input_block="There were a couple of minor hiccups along the way, but all in all, I'm quite pleased with the final result.",
│   │   label=True
│   ),
│   Sample(
│   │   reasoning="The sentence begins with positive praise regarding the elegant packaging and timely delivery, but pivots with the contrastive conjunction 'but' to highlight a fatal product defect that completely ruins the experience, making the overall sentiment negative.",
│   │   input_block='The packaging was elegant and arrived right on schedule, but the device inside was completely cracked and failed to power on at all.',
│   │   label=False
│   ),
│   Sample(
│   │   reasoning='The sentence expresses a clearly positive sentiment through a behavioral endorsement (repurchasing multiple units for different locations), despite relying entirely on factual, action-oriented language without explicit positive emotional keywords or evaluative adjectives.',
│   │   input_block='I ended up ordering two more to keep at my office and in my travel bag.',
│   │   label=True
│   ),
│   Sample(
│   │   reasoning="The sentence uses double negative and negation constructs ('not a single aspect ... failed to exceed' and 'cannot recommend it enough') which syntactically involve negative words ('not', 'failed', 'cannot') but semantically resolve to a strong positive endorsement and praise of the product.",
│   │   input_block='There is not a single aspect of this coffee maker that failed to exceed my expectations, and I cannot recommend it enough.',
│   │   label=True
│   )
]
```

- Model: gemini-3.7-flash
- Took: 1m20s
- Usage: 73194 tokens (44347 in, 8055 out, 0 cached).
- Price: $0.0635

## Bonus — Observe the events

Pass an observer to the `LLM` and you see everything that happens: decomposition, draws, debates, refinements, LLM calls.

By default, a silent observer is used (NullObserver).
A logging observer is shipped with Barred, see below.
You can implement your own by inheriting from NullObserver.

In [ ]:
import logging
import sys

from barred import LLM, LoggingObserver

# LoggingObserver writes to the "barred" logger, silent until it is given a handler.
# INFO = the milestones, DEBUG = the prompts and the responses.
logging.getLogger("barred").handlers = [logging.StreamHandler(sys.stdout)]
logging.getLogger("barred").setLevel(logging.INFO)
logging.getLogger("barred").propagate = False

llm = LLM(
    provider=provider,
    model=model,
    observer=LoggingObserver(),  # <-- the observer
    **kwargs,
)

# Now use the lib
# decompose_dimensions(llm, ...)
# barred(llm, ...)